<br><br><br><br>

# 🟩 문제 풀기 2

- 주급 계산
  - 주급 관련 테이블은 각자 SQL에서 알아서 만드세요.
  - 아이디(id), 이름(wname), 근무시간(work_time), 시간당급여액(per_pay)
  - 연장수당(overtime_pay) - case when문 근무 시간 > 20

## 🟢 1. 주급 관련 테이블 만들기

In [ ]:
CREATE TABLE tb_weekly_pay(
					id bigint PRIMARY KEY AUTO_INCREMENT,
					wname varchar(20) NOT NULL,
					work_time int NOT NULL,
					per_pay int NOT NULL,
					overtime_pay Int NOT NULL,
					weekly_pay Int NOT NULL,
					regdate datetime
);

INSERT INTO tb_weekly_pay(wname, work_time,  per_pay, overtime_pay, weekly_pay, regdate)
values('홍길동', 10, 10000, 0, 100000, now());

SELECT 
					id, wname, work_time,  per_pay, overtime_pay, weekly_pay, 
					date_format(regdate, '%Y-%m-%d %H:%i') regdate
FROM tb_weekly_pay;

## 🟢 2. Python에서 MySQL 접근하여 python 코드로 control하기
직접 만들어보기

In [ ]:
import pymysql
import pymysql.cursors


class MysqlWeeklyPay:
    def __init__(self):
        self.conn = self.mysql_conn()
        self.cursor = self.conn.cursor(pymysql.cursors.DictCursor)

    def mysql_conn(self):
        conn = pymysql.connect(
            host="localhost",
            user="root",
            password="",
            db="mydb",
            port=3306,
        )
        print("접속 성공")
        return conn

    def all_view_data(self): # 1
        sql = """
            SELECT 
                id, wname, work_time,  per_pay, overtime_pay, weekly_pay, 
                date_format(regdate, '%Y-%m-%d %H:%i') regdate
            FROM tb_weekly_pay;
        """
        print(sql)
        self.cursor.execute(sql)
        rows = self.cursor.fetchall()
        print("데이터 개수", len(rows))
        for row in rows:
            print(
                row["id"],
                row["wname"],
                row["work_time"],
                row["per_pay"],
                row["overtime_pay"],
                row["weekly_pay"],
            )
        print()

    def calcul_overtime(self, work_time, per_pay):
        if work_time > 20:
          value = work_time - 20 
          return value * (per_pay * 1.5)
        else:
          return 0

    def calcul_weekly_pay(self, work_time, per_pay, overtime_pay):
        if overtime_pay == 0:
          return work_time * per_pay
        else:
          return (20 * per_pay) + overtime_pay

    def insert_data(self): # 2
        wname = input("이름 : ")
        work_time = int(input("근무시간? : "))
        per_pay = int(input("시급 얼마? : "))
        overtime_pay = self.calcul_overtime(work_time, per_pay)
        weekly_pay = self.calcul_weekly_pay(work_time, per_pay, overtime_pay)

        sql = """
            INSERT INTO tb_weekly_pay(wname, work_time,  per_pay, overtime_pay, weekly_pay, regdate)
            values(%s, %s, %s, %s, %s, now())
        """
        self.cursor.execute(sql, (wname, work_time, per_pay, overtime_pay, weekly_pay))
        self.conn.commit()
        print("INSERT 완료")

        self.cursor.execute("SELECT * FROM tb_weekly_pay ORDER BY id DESC LIMIT 1")
        row = self.cursor.fetchone()
        print("방금 추가된 데이터:", row)
        print()

    def update_data(self):
        self.all_view_data()

        id = input("수정할 id값 입력 : ")
        wname = input("이름 : ")
        work_time = int(input("근무시간? : "))
        per_pay = int(input("시급 얼마? : "))
        overtime_pay = self.calcul_overtime(work_time, per_pay)
        weekly_pay = self.calcul_weekly_pay(work_time, per_pay, overtime_pay)

        sql = """
            UPDATE tb_weekly_pay
            SET 
                wname = %s,
                work_time = %s,
                per_pay = %s,
                overtime_pay = %s,
                weekly_pay = %s
            WHERE id = %s
        """
        self.cursor.execute(sql, (wname, work_time, per_pay, overtime_pay, weekly_pay, id))
        self.conn.commit()
        print("수정 완료")

        self.all_view_data()
        print()

    def delete_data(self):
        self.all_view_data()

        sname = input("삭제할 이름을 입력하세요 : ")
        sql = """
            DELETE FROM tb_weekly_pay WHERE wname = %s
        """
        self.cursor.execute(sql, sname)
        self.conn.commit()
        print("삭제 완료")

        self.all_view_data()
        print()

    def start(self):
        while True:
            print(f"1.전체보기  |  2.추가  |  3.수정  |  4.삭제  |  0.종료")
            select = input("🔢 번호 선택: ")

            if select == "1":
                self.all_view_data()
            elif select == "2":
                self.insert_data()
            elif select == "3":
                self.update_data()
            elif select == "4":
                self.delete_data()
            elif select == "0":
                break


if __name__ == "__main__":
    m = MysqlWeeklyPay()
    m.start()


접속 성공
1.전체보기  |  2.추가  |  3.수정  |  4.삭제  |  0.종료



            SELECT 
                id, wname, work_time,  per_pay, overtime_pay, weekly_pay, 
                date_format(regdate, '%Y-%m-%d %H:%i') regdate
            FROM tb_weekly_pay;
        
데이터 개수 2
1 홍길동 10 10000 0 100000
2 김민지 24 10000 60000 260000

수정 완료

            SELECT 
                id, wname, work_time,  per_pay, overtime_pay, weekly_pay, 
                date_format(regdate, '%Y-%m-%d %H:%i') regdate
            FROM tb_weekly_pay;
        
데이터 개수 2
1 홍길동 10 10000 0 100000
2 김수민 19 10000 0 190000


1.전체보기  |  2.추가  |  3.수정  |  4.삭제  |  0.종료

            SELECT 
                id, wname, work_time,  per_pay, overtime_pay, weekly_pay, 
                date_format(regdate, '%Y-%m-%d %H:%i') regdate
            FROM tb_weekly_pay;
        
데이터 개수 2
1 홍길동 10 10000 0 100000
2 김수민 19 10000 0 190000

삭제 완료

            SELECT 
                id, wname, work_time,  per_pay, overtime_pay, weekly_pay, 
                date_format(regdate, '%Y-%m-%d %H:%i') regdate
       